# Switchyard Business Client

This notebook verifies the Switchyard endpoint and builds a LangChain customer-request triage app. Every invocation uses the public route `model="switchyard"`. The two examples demonstrate **Efficient** for a routine request and **Capable** after a simulated critical diagnostic failure, and print the deployment actually selected by Switchyard.


## 1. Install the LangChain OpenAI integration

Switchyard exposes an OpenAI-compatible Chat Completions API, so the app uses LangChain's `ChatOpenAI` integration with a custom base URL.

In [ ]:
%pip install -qU langchain-openai

## 2. Connect to your Container App

The setup first reads `SWITCHYARD_FQDN` from the notebook environment. If it is not set, it resolves the hostname with Azure CLI using `RESOURCE_GROUP` and `SWITCHYARD_APP_NAME`.

Complete the lab README's Container App setup first. The Foundry API key is stored in an ACA secret and supplied to Switchyard through its `AZURE_AI_API_KEY` environment variable. With `forward_auth = false` and `api_key_env = "AZURE_AI_API_KEY"`, Switchyard authenticates upstream requests using that server-held key; this notebook does not read, request, or send the Foundry key.

This ephemeral Skillable lab uses a public HTTPS endpoint without caller authentication. Anyone who knows the URL can make requests charged against the Foundry key stored in ACA until the app is deleted. Keep the URL within the lab and delete your app when finished. Do not paste Foundry keys into notebook cells or outputs.

In [ ]:
import os
import subprocess


def run_az(*arguments: str) -> str:
    """Run Azure CLI and return a non-empty text result."""
    result = subprocess.run(
        ["az", *arguments],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
    if not result:
        raise RuntimeError(f"Azure CLI returned no value for: az {' '.join(arguments)}")
    return result


switchyard_fqdn = os.getenv("SWITCHYARD_FQDN", "").strip()
if not switchyard_fqdn:
    resource_group = os.getenv("RESOURCE_GROUP", "").strip()
    switchyard_app_name = os.getenv("SWITCHYARD_APP_NAME", "").strip()
    if not resource_group or not switchyard_app_name:
        raise RuntimeError(
            "Set SWITCHYARD_FQDN, or set both RESOURCE_GROUP and "
            "SWITCHYARD_APP_NAME, before running this cell."
        )
    switchyard_fqdn = run_az(
        "containerapp",
        "show",
        "--name",
        switchyard_app_name,
        "--resource-group",
        resource_group,
        "--query",
        "properties.configuration.ingress.fqdn",
        "--output",
        "tsv",
    )

# The portable curl test needs only the hostname, not a Foundry credential.
os.environ["SWITCHYARD_FQDN"] = switchyard_fqdn

switchyard_base_url = f"https://{switchyard_fqdn}/v1"
print(f"Switchyard endpoint: {switchyard_base_url}")
print("Public endpoint without caller authentication; requests use the Foundry key stored in ACA. Delete the app after the lab.")

## 3. Verify the endpoint with curl

This is the same client test used in the lab README. A successful response confirms that the public Switchyard route is available and the upstream deployment accepts the server-held Foundry key. No client Authorization header is required. The Python cell invokes `curl.exe` on Windows or `curl` on other systems; Bash is not required.

In [ ]:
import json
import shutil
import subprocess

curl_executable = shutil.which("curl.exe") or shutil.which("curl")
if not curl_executable:
    raise RuntimeError("curl is required for this notebook test.")

curl_payload = {
    "model": "switchyard",
    "messages": [
        {
            "role": "user",
            "content": "Explain model routing in two sentences.",
        }
    ],
}
curl_response = subprocess.run(
    [
        curl_executable,
        "--fail-with-body",
        "--silent",
        "--show-error",
        "--request",
        "POST",
        f"{switchyard_base_url}/chat/completions",
        "--header",
        "Content-Type: application/json",
        "--data-binary",
        "@-",
    ],
    input=json.dumps(curl_payload),
    capture_output=True,
    text=True,
    encoding="utf-8",
)
print(curl_response.stdout, end="")
if curl_response.stderr:
    print(curl_response.stderr, end="")
curl_response.check_returncode()

## 4. Build a LangChain customer-request triage app

The chain turns a customer message and optional diagnostic history into a Markdown business brief. It retains the LangChain `AIMessage` so routing metadata is available alongside the answer; a `StrOutputParser` would discard that metadata.

`include_response_headers=True` exposes Switchyard's `x-model-router-selected-model` header in `response.response_metadata["headers"]`. This is the selected target's configured model/deployment ID. The response body's `model` is also shown as `response_metadata["model_name"]`; Foundry may report a catalog model/version rather than your deployment name. Neither value comes from asking the model to identify itself.

The OpenAI SDK requires an `api_key` value, so this client uses the literal non-secret placeholder `not-used`. It does not authenticate the caller. With `forward_auth = false`, Switchyard uses the Foundry key stored in ACA instead. Delete the public lab app when finished.


In [ ]:
from IPython.display import Markdown, display
from langchain_core.messages import AIMessage, ToolMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import ChatOpenAI
from openai import NotFoundError


model = ChatOpenAI(
    model="switchyard",
    base_url=switchyard_base_url,
    api_key="not-used",
    use_responses_api=False,
    include_response_headers=True,
    # Omit temperature so each routed model uses its supported default.
    temperature=None,
)

triage_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """You are a customer-operations analyst. Triage incoming business requests
for a human support team. Do not invent policies, commitments, refunds, or
technical facts. Clearly label any assumptions.

Return concise Markdown with exactly these headings:
## Priority
Choose Low, Medium, High, or Critical and give one-sentence reasoning.
## Owning team
Choose Sales, Billing, Technical Support, Security, or Customer Success.
## Summary
Summarize the request in no more than three bullets.
## Recommended actions
Give a numbered list of the next actions for the business.
## Draft response
Write a concise, professional response to the customer.""",
        ),
        (
            "human",
            """Customer: {customer_name}
Account tier: {account_tier}
Request received through: {channel}

Customer message:
{customer_message}""",
        ),
        MessagesPlaceholder("diagnostic_history", optional=True),
    ]
)

triage_chain = triage_prompt | model


def triage_customer_request(
    customer_name: str,
    account_tier: str,
    channel: str,
    customer_message: str,
    diagnostic_history: list | None = None,
) -> AIMessage:
    """Analyze one customer request through the Switchyard route."""
    try:
        return triage_chain.invoke(
            {
                "customer_name": customer_name,
                "account_tier": account_tier,
                "channel": channel,
                "customer_message": customer_message,
                "diagnostic_history": diagnostic_history or [],
            }
        )
    except NotFoundError as error:
        raise RuntimeError(
            "Switchyard returned HTTP 404: a route or upstream deployment was "
            "not found or is inaccessible. If step 5 succeeded and step 6 failed, "
            "check the Capable deployment ID in the deployed routes-config "
            "against the Foundry deployment list (not the model catalog name), "
            "its provisioning state, and access through the configured Foundry "
            "account. Updating a local environment variable does not update "
            "the app's stored routing configuration. See the README section "
            "'Step 6 returns HTTP 404'. No successful model selection can be "
            "reported for this failed request."
        ) from error


def selected_deployment(response: AIMessage) -> str:
    """Read the server's routing result, never infer it from answer text."""
    headers = {
        name.lower(): value
        for name, value in response.response_metadata.get("headers", {}).items()
    }
    deployment = headers.get("x-model-router-selected-model")
    if not deployment:
        raise RuntimeError(
            "No Switchyard selected-model header was returned. Check that the "
            "app runs the README's pinned Switchyard version and preserves "
            "response headers; model_name alone does not confirm the selected target."
        )
    return deployment


def show_result(response: AIMessage, expected_tier: str) -> str:
    """Print routing evidence and render the business brief."""
    deployment = selected_deployment(response)
    print(f"Expected tier for this example: {expected_tier}")
    print("Requested route: switchyard")
    print(f"Switchyard selected model/deployment: {deployment}")
    print(f"Provider-reported model: {response.response_metadata.get('model_name', 'unavailable')}")
    display(Markdown(response.content))
    return deployment


## 5. Routine request → Efficient

The README configures Switchyard v0.3.0 with `stage_router`, `picker="efficient_first"`, and `confidence_threshold=0.5`, without an LLM task classifier. This router examines tool-result signals, **not prompt complexity or business urgency**. A new request without tool history falls back to Efficient. Changing only the prose to a harder question does not reliably select Capable.

Run this routine billing request and inspect the printed deployment ID. Compare it with `EFFICIENT_MODEL_DEPLOYMENT` used in the README's routing configuration.


In [ ]:
routine_result = triage_customer_request(
    customer_name="Contoso Retail",
    account_tier="Standard",
    channel="Email",
    customer_message=(
        "Please email me a copy of last month's invoice. "
        "There are no disputed charges or service issues."
    ),
)

efficient_deployment = show_result(routine_result, expected_tier="Efficient")


## 6. Critical diagnostic failure → Capable

This example replays a **simulated** diagnostic tool exchange using LangChain's `AIMessage` and `ToolMessage`. No diagnostic tool is executed and no real incident is reported. The assistant tool call and matching tool result form valid Chat Completions history; the selected model continues that history to produce the brief.

The tool result contains `fatal runtime error: out of memory`, a critical-error signal recognized by the pinned stage router. That signal forces Capable regardless of the confidence threshold. A critical error mentioned only in the customer message would not exercise this tool-result rule. We still request `model="switchyard"`; Switchyard makes the selection without changing routes or deployments in the client.

Compare the selected ID with `CAPABLE_MODEL_DEPLOYMENT` from the README. Use two **different** deployment IDs for the lab to demonstrate different models.

**If step 5 works but this step returns HTTP 404:** check the Capable target's deployed configuration and Foundry deployment availability. The notebook cannot repair a missing or inaccessible deployment by changing the prompt. Follow **Step 6 returns HTTP 404** in the lab README; do not substitute the Efficient target just to make the comparison pass. Switchyard attaches the selected-model header only to successful responses, so this failed call cannot print a confirmed selection.


In [ ]:
simulated_diagnostics = [
    AIMessage(
        content="I will review the order-processing diagnostic result before triaging.",
        tool_calls=[
            {
                "name": "Bash",
                "args": {"command": "python diagnose_orders.py"},
                "id": "call_demo_diagnostics",
                "type": "tool_call",
            }
        ],
    ),
    ToolMessage(
        content=(
            "SIMULATED LAB DIAGNOSTIC OUTPUT (no command was executed):\n"
            "fatal runtime error: out of memory\n"
            "Order-processing worker terminated; service recovery needs investigation."
        ),
        tool_call_id="call_demo_diagnostics",
    ),
]

incident_result = triage_customer_request(
    customer_name="Contoso Retail",
    account_tier="Enterprise",
    channel="Email",
    customer_message=(
        "Our production order-processing integration has intermittent timeouts "
        "affecting 20% of orders. A rollback did not resolve the errors. "
        "Use the simulated diagnostic history to propose a recovery plan, "
        "identify missing evidence, and draft a customer update. "
        "Do not execute tools or claim that recovery actions have been completed."
    ),
    diagnostic_history=simulated_diagnostics,
)

capable_deployment = show_result(incident_result, expected_tier="Capable")


## 7. Compare the actual routing results

The table uses the selected-model response headers from the two invocations. The tier labels describe the expected paths under the README configuration. The check fails if both requests returned the same deployment, rather than claiming that both tiers were demonstrated. If it fails, check that the efficient and capable targets use distinct deployment IDs, the app uses the README's stage-router settings and pinned version, and the simulated tool history was included.

You can edit the requests and rerun the examples. With this stage router, different natural-language complexity alone does not imply different model choices. For routing based on task text, a server-side task classifier would need to be configured separately.


In [ ]:
display(Markdown(
    "| Invocation | Expected tier | Selected deployment (response header) |\n"
    "| --- | --- | --- |\n"
    f"| Routine billing request | Efficient | `{efficient_deployment}` |\n"
    f"| Critical diagnostic failure | Capable | `{capable_deployment}` |"
))

if efficient_deployment == capable_deployment:
    raise RuntimeError(
        "Both invocations selected the same deployment. Check distinct target "
        "IDs, the stage-router configuration, and the simulated tool history."
    )
print("Confirmed: the two LangChain invocations were served by different deployments.")
